# Human Turing Test Results

Analysis of human rater sessions from `results/`. Each rater sees 10 conversations (5 real + 5 simulated)

**Primary metrics**:
- **Accuracy**: overall fraction of correctly classified conversations $(TP + TN) / N$
- **Real recognition rate**: fraction of real conversations correctly identified as real $TN / (TN + FP)$
- **Simulation detection rate**: fraction of simulated conversations correctly identified $TP / (TP + FN)$

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy import stats

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.grid"] = False
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"

SIM_COLOR = "#E74C3C"
REAL_COLOR = "#2E86C1"

RESULTS_DIR = Path("results")

EXPORT_DIR = Path("..") / "report" / "data"


In [ ]:
# Load all session files
all_ratings = []
session_meta = []

EXCLUDE = {("rater_0", None), ("rater_13", "6a793617-ea04-4c6f-8897-26e17a59a705")}

for path in sorted(RESULTS_DIR.glob("session_*.json")):
    with open(path) as f:
        session = json.load(f)
    rater_name = session.get("rater", "anonymous")
    rater_index = session.get("rater_index")
    rater_id = f"rater_{rater_index}" if rater_index is not None else rater_name

    if (rater_id, None) in EXCLUDE:
        continue

    kept = 0
    for r in session["ratings"]:
        if (rater_id, r["conversation_id"]) in EXCLUDE:
            continue
        all_ratings.append({
            "rater_id": rater_id,
            "rater_index": rater_index,
            "conversation_id": r["conversation_id"],
            "ground_truth": r["ground_truth"],
            "classification": r["classification"],
            "classified_simulated": r["classified_simulated"],
            "correct": r["correct"],
            "feedback": r.get("feedback", ""),
            "time_seconds": r.get("time_seconds"),
            "presentation_order": r["presentation_order"],
        })
        kept += 1

    session_meta.append({
        "rater_id": rater_id,
        "rater_name": rater_name,
        "rater_index": rater_index,
        "n_ratings": kept,
        "file": path.name,
    })

df = pd.DataFrame(all_ratings)
meta_df = pd.DataFrame(session_meta)

print(f"Loaded {len(df)} ratings from {df['rater_id'].nunique()} raters across {len(meta_df)} sessions")
print(f"  Simulated judgements: {(df['ground_truth'] == 'simulated').sum()}")
print(f"  Real judgements: {(df['ground_truth'] == 'real').sum()}")
print()
meta_df

## Primary Analysis

In [ ]:
def wilson_ci(k, n, alpha=0.05):
    """Two-sided Wilson score interval for a proportion."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    z = stats.norm.ppf(1 - alpha / 2)
    denom = 1.0 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = (z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / denom
    return max(centre - half, 0.0), min(centre + half, 1.0)

def wilson_upper(k, n, alpha=0.05):
    """One-sided Wilson score upper bound for a proportion."""
    if n == 0:
        return np.nan
    p = k / n
    z = stats.norm.ppf(1 - alpha)
    denom = 1.0 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = (z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / denom
    return min(centre + half, 1.0)


def compute_protocol_metrics(subset_df, label):
    sim = subset_df[subset_df["ground_truth"] == "simulated"]
    real = subset_df[subset_df["ground_truth"] == "real"]

    tp = sim["classified_simulated"].sum()
    fn = len(sim) - tp
    fp = real["classified_simulated"].sum()
    tn = len(real) - fp
    n_sim = len(sim)
    n_real = len(real)

    det_rate = tp / n_sim if n_sim else 0
    fooled_rate = fn / n_sim if n_sim else 0
    real_recog_rate = tn / n_real if n_real else 0
    false_accusation = fp / n_real if n_real else 0
    accuracy = (tp + tn) / (tp + fn + fp + tn) if (tp + fn + fp + tn) else 0
    balanced_acc = 0.5 * det_rate + 0.5 * real_recog_rate

    ci_lo, ci_hi = wilson_ci(tp, n_sim)
    upper_one_sided = wilson_upper(tp, n_sim)

    acc_ci_lo, acc_ci_hi = wilson_ci(int(tp + tn), int(tp + fn + fp + tn))
    rr_ci_lo, rr_ci_hi = wilson_ci(int(tn), int(n_real))

    return {
        "label": label,
        "n_raters": subset_df["rater_id"].nunique(),
        "n_sim": n_sim,
        "n_real": n_real,
        "tp": tp, "fn": fn, "fp": fp, "tn": tn,
        "det_rate": det_rate,
        "fooled_rate": fooled_rate,
        "real_recog_rate": real_recog_rate,
        "false_accusation": false_accusation,
        "accuracy": accuracy,
        "balanced_acc": balanced_acc,
        "ci_95_lo": ci_lo,
        "ci_95_hi": ci_hi,
        "upper_one_sided_95": upper_one_sided,
        "acc_ci_lo": acc_ci_lo,
        "acc_ci_hi": acc_ci_hi,
        "rr_ci_lo": rr_ci_lo,
        "rr_ci_hi": rr_ci_hi,
    }

results = compute_protocol_metrics(df, "Human raters")
results_df = pd.DataFrame([results])

def fmt_pct_ci(val, lo, hi):
    return f"{val:.1%} ({lo:.1%}, {hi:.1%})"

r = results
summary = pd.DataFrame([{
    "Raters": int(r["n_raters"]),
    "N (sim)": int(r["n_sim"]),
    "N (real)": int(r["n_real"]),
    "Accuracy (95% CI)": fmt_pct_ci(r["accuracy"], r["acc_ci_lo"], r["acc_ci_hi"]),
    "Sim. Detection Rate (95% CI)": fmt_pct_ci(r["det_rate"], r["ci_95_lo"], r["ci_95_hi"]),
    "Real Recognition Rate (95% CI)": fmt_pct_ci(r["real_recog_rate"], r["rr_ci_lo"], r["rr_ci_hi"]),
}])
summary

# display_cols = [
#     "label", "n_raters", "n_sim", "n_real",
#     "accuracy", "acc_ci_lo", "acc_ci_hi",
#     "real_recog_rate", "rr_ci_lo", "rr_ci_hi",
#     "det_rate", "ci_95_lo", "ci_95_hi", "upper_one_sided_95",
#     "fooled_rate", "balanced_acc",
#     "tp", "fn", "fp", "tn",
# ]
# styled = results_df[display_cols].copy()
# for col in ["accuracy", "acc_ci_lo", "acc_ci_hi",
#             "real_recog_rate", "rr_ci_lo", "rr_ci_hi",
#             "det_rate", "ci_95_lo", "ci_95_hi", "upper_one_sided_95",
#             "fooled_rate", "balanced_acc"]:
#     styled[col] = styled[col].map(lambda x: f"{x:.1%}")
# styled.columns = [
#     "Analysis", "Raters", "N(sim)", "N(real)",
#     "Accuracy", "Acc CI low", "Acc CI high",
#     "Real Recog.", "RR CI low", "RR CI high",
#     "Det. Rate", "DR CI low", "DR CI high", "DR Upper 1-sided",
#     "Fooled Rate", "Balanced Acc",
#     "TP", "FN", "FP", "TN",
# ]
# styled

In [ ]:
r = results
tp = int(r["tp"])
tn = int(r["tn"])
n_sim = int(r["n_sim"])
n_real = int(r["n_real"])
n_total = n_sim + n_real
n_correct = tp + tn
det = r["det_rate"]
ub = r["upper_one_sided_95"]
acc = r["accuracy"]
rr = r["real_recog_rate"]

print(f"=== Human Raters ({int(r['n_raters'])} raters) ===")
print(f"Overall accuracy: {n_correct}/{n_total} = {acc:.1%} (correctly classified across all conversations).")
print(f"Real recognition rate: {tn}/{n_real} = {rr:.1%} (real conversations correctly identified as real).")
print(f"Simulation detection rate: {tp}/{n_sim} = {det:.1%} (simulated conversations correctly identified).")
print(f"One-sided 95% upper confidence bound on detection rate: {ub:.1%}.")
print()
print(f"Raters were better at detecting simulated conversations ({det:.1%}) than recognising real ones ({rr:.1%}),")
print(f"with {int(r['fp'])} of {n_real} real conversations falsely accused of being simulated.")

## Per-Rater Breakdown

In [ ]:
rater_rows = []
for rater_id, rater_data in df.groupby("rater_id"):
    sim = rater_data[rater_data["ground_truth"] == "simulated"]
    real = rater_data[rater_data["ground_truth"] == "real"]
    tp = int(sim["classified_simulated"].sum())
    fn = len(sim) - tp
    fp = int(real["classified_simulated"].sum())
    tn = len(real) - fp
    n_total = tp + fn + fp + tn
    accuracy = (tp + tn) / n_total if n_total else 0
    rater_idx = rater_data.iloc[0]["rater_index"]
    rater_rows.append({
        "Rater": rater_id,
        "Index": int(rater_idx) if rater_idx is not None else None,
        "Accuracy": f"{tp+tn}/{n_total} = {accuracy:.0%}",
        "Real Recognition Rate": f"{tn}/{len(real)} = {tn/len(real):.0%}" if len(real) else "-",
        "Simulation Detection Rate": f"{tp}/{len(sim)} = {tp/len(sim):.0%}" if len(sim) else "-",
        "TP": tp, "FN": fn, "FP": fp, "TN": tn,
    })

rater_table = pd.DataFrame(rater_rows)
rater_table = rater_table.sort_values("Index").reset_index(drop=True)
rater_table

In [ ]:
# Confusion matrix as figure
fig, ax = plt.subplots(figsize=(4.5, 4))
cmap = plt.cm.GnBu
labels = ["Real", "Simulated"]

r = results
cm = np.array([[r["tn"], r["fp"]], [r["fn"], r["tp"]]])
im = ax.imshow(cm, cmap=cmap, vmin=0, vmax=cm.max() or 1, aspect="equal")
for i in range(2):
    for j in range(2):
        total_row = cm[i].sum()
        pct = cm[i, j] / total_row * 100 if total_row else 0
        ax.text(j, i, f"{cm[i, j]}\n({pct:.0f}%)",
                ha="center", va="center", fontsize=12,
                color="white" if cm[i, j] > cm.max() * 0.6 else "black")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("Classified", fontsize=11)
ax.set_ylabel("Ground Truth", fontsize=11)
ax.set_title("Human Rater Confusion Matrix", fontsize=12)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False, bottom=False)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "human_confusion_matrix.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Per-rater bar plot: % judged as real, split by ground truth
raters = sorted(df["rater_id"].unique(),
                key=lambda r: int(r.rsplit("_", 1)[-1]) if r.rsplit("_", 1)[-1].isdigit() else 0)

fig, ax = plt.subplots(figsize=(max(12, len(raters) * 1.1), 5))

x = np.arange(len(raters))
width = 0.3

sim_judged_real = []
real_judged_real = []
for rater in raters:
    rater_data = df[df["rater_id"] == rater]
    sim_data = rater_data[rater_data["ground_truth"] == "simulated"]
    real_data = rater_data[rater_data["ground_truth"] == "real"]
    sim_judged_real.append((~sim_data["classified_simulated"]).sum() / len(sim_data) if len(sim_data) else 0)
    real_judged_real.append((~real_data["classified_simulated"]).sum() / len(real_data) if len(real_data) else 0)

ax.bar(x - width / 2, sim_judged_real, width, color=SIM_COLOR)
ax.bar(x + width / 2, real_judged_real, width, color=REAL_COLOR)

for i, (sv, rv) in enumerate(zip(sim_judged_real, real_judged_real)):
    ax.text(x[i] - width / 2, sv + 0.02, f"{sv:.0%}", ha="center", fontsize=7)
    ax.text(x[i] + width / 2, rv + 0.02, f"{rv:.0%}", ha="center", fontsize=7)

ax.axhline(0.5, color="grey", linestyle="--", alpha=0.5)
short_labels = [str(int(r.rsplit("_", 1)[-1])) if r.rsplit("_", 1)[-1].isdigit() else r for r in raters]
ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=9)
ax.set_xlabel("Rater")
ax.set_ylabel("% judged as real")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.set_ylim(-0.05, 1.15)

for spine in ax.spines.values():
    spine.set_edgecolor("#AAAAAA")
    spine.set_linewidth(0.7)
ax.tick_params(axis="both", width=0.7)

fig.legend(
    handles=[
        Patch(facecolor=SIM_COLOR, label="simulated conversations"),
        Patch(facecolor=REAL_COLOR, label="real conversations"),
        Line2D([0], [0], color="grey", linestyle="--", alpha=0.5, label="chance"),
    ],
    loc="upper center", ncol=3, frameon=True, fontsize=10,
    bbox_to_anchor=(0.5, -0.01),
)
fig.suptitle("Per-Rater: % of Conversations Judged as Real (Human Raters)", fontsize=13)
plt.tight_layout(rect=[0, 0.05, 1, 1])

plt.savefig(EXPORT_DIR / "human_per_rater.pdf", bbox_inches="tight", dpi=150)
plt.show()

## Confidence Intervals

In [ ]:
def _ci_for_metric(k, n):
    lo, hi = wilson_ci(int(k), int(n))
    return k / n if n else 0, lo, hi

r = results
sim_rate, sim_lo, sim_hi = _ci_for_metric(r["tp"], r["n_sim"])
real_rate, real_lo, real_hi = _ci_for_metric(r["tn"], r["n_real"])
total_correct = int(r["tp"]) + int(r["tn"])
total_n = int(r["n_sim"]) + int(r["n_real"])
overall_rate, overall_lo, overall_hi = _ci_for_metric(total_correct, total_n)

fig, axes = plt.subplots(1, 3, figsize=(16, 3), sharey=True)

metrics = [
    (sim_rate, sim_lo, sim_hi, "Simulation Detection Rate\n(simulated conversations only)"),
    (real_rate, real_lo, real_hi, "Real Recognition Rate\n(real conversations only)"),
    (overall_rate, overall_lo, overall_hi, "Overall Accuracy\n(all conversations)"),
]

for ax, (rate, lo, hi, title) in zip(axes, metrics):
    ax.barh(0, rate, height=0.5, color=REAL_COLOR, alpha=0.8, zorder=2)
    ax.errorbar(rate, 0, xerr=[[rate - lo], [hi - rate]],
                fmt="none", ecolor="#888888", capsize=3, linewidth=0.8, zorder=3)
    ax.text(hi + 0.02, 0, f"{rate:.0%} [{lo:.0%}, {hi:.0%}]", va="center", fontsize=8)

    ax.axvline(0.50, color="grey", linestyle=":", linewidth=0.8, alpha=0.4)
    ax.set_yticks([0])
    ax.set_yticklabels(["Human"])
    ax.set_xlim(0, 1.0)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.set_title(title, fontsize=10)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

fig.suptitle("Human Raters — 95% Confidence Intervals", fontsize=13)
plt.tight_layout()

plt.savefig(EXPORT_DIR / "human_ci.pdf", bbox_inches="tight", dpi=150)
plt.show()

## Learning Effect: Do Metrics Improve with Exposure?

In [ ]:
max_pos = int(df["presentation_order"].max())
positions = range(1, max_pos + 1)

ACC_COLOR = "#27AE60"

metrics_by_pos = {"det_rate": [], "det_lo": [], "det_hi": [],
                  "rr_rate": [], "rr_lo": [], "rr_hi": [],
                  "acc_rate": [], "acc_lo": [], "acc_hi": []}

for k in positions:
    up_to_k = df[df["presentation_order"] <= k]
    sim_k = up_to_k[up_to_k["ground_truth"] == "simulated"]
    real_k = up_to_k[up_to_k["ground_truth"] == "real"]

    # Detection rate
    n_sim = len(sim_k)
    if n_sim > 0:
        tp = int(sim_k["classified_simulated"].sum())
        metrics_by_pos["det_rate"].append(tp / n_sim)
        lo, hi = wilson_ci(tp, n_sim)
        metrics_by_pos["det_lo"].append(lo)
        metrics_by_pos["det_hi"].append(hi)
    else:
        for key in ["det_rate", "det_lo", "det_hi"]:
            metrics_by_pos[key].append(np.nan)

    # Real recognition rate
    n_real = len(real_k)
    if n_real > 0:
        tn = int((~real_k["classified_simulated"]).sum())
        metrics_by_pos["rr_rate"].append(tn / n_real)
        lo, hi = wilson_ci(tn, n_real)
        metrics_by_pos["rr_lo"].append(lo)
        metrics_by_pos["rr_hi"].append(hi)
    else:
        for key in ["rr_rate", "rr_lo", "rr_hi"]:
            metrics_by_pos[key].append(np.nan)

    # Overall accuracy
    n_all = len(up_to_k)
    if n_all > 0:
        correct = int(up_to_k["correct"].sum())
        metrics_by_pos["acc_rate"].append(correct / n_all)
        lo, hi = wilson_ci(correct, n_all)
        metrics_by_pos["acc_lo"].append(lo)
        metrics_by_pos["acc_hi"].append(hi)
    else:
        for key in ["acc_rate", "acc_lo", "acc_hi"]:
            metrics_by_pos[key].append(np.nan)

x = list(positions)

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.fill_between(x, metrics_by_pos["det_lo"], metrics_by_pos["det_hi"], alpha=0.10, color=SIM_COLOR)
ax.plot(x, metrics_by_pos["det_rate"], "-o", color=SIM_COLOR, markersize=5, linewidth=2, zorder=3, label="Simulation detection rate")

ax.fill_between(x, metrics_by_pos["rr_lo"], metrics_by_pos["rr_hi"], alpha=0.10, color=REAL_COLOR)
ax.plot(x, metrics_by_pos["rr_rate"], "-s", color=REAL_COLOR, markersize=5, linewidth=2, zorder=3, label="Real recognition rate")

ax.fill_between(x, metrics_by_pos["acc_lo"], metrics_by_pos["acc_hi"], alpha=0.10, color=ACC_COLOR)
ax.plot(x, metrics_by_pos["acc_rate"], "-^", color=ACC_COLOR, markersize=5, linewidth=2, zorder=3, label="Overall accuracy")

ax.axhline(0.50, color="grey", linestyle=":", linewidth=0.8, alpha=0.4, label="Chance")

ax.set_xticks(x)
ax.set_xlabel("Presentation order (position)")
ax.set_ylabel("Cumulative rate")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.set_ylim(-0.05, 1.05)
ax.set_xlim(0.5, max_pos + 0.5)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.legend(loc="upper right", fontsize=9, frameon=True)
ax.set_title("Learning Effect: Cumulative Metrics by Presentation Order\n"
             "(95% Wilson CI)", fontsize=13)
plt.tight_layout()

plt.savefig(EXPORT_DIR / "human_learning_effect.pdf", bbox_inches="tight", dpi=150)
plt.show()

## Time Metrics

In [ ]:
time_df = df.dropna(subset=["time_seconds"])

def mean_ci_95(series):
    n = len(series)
    if n < 2:
        return series.mean(), np.nan, np.nan
    mean = series.mean()
    sem = series.sem()
    t_crit = stats.t.ppf(0.975, n - 1)
    return mean, mean - t_crit * sem, mean + t_crit * sem

def fmt_mean_ci(series):
    mean, lo, hi = mean_ci_95(series)
    if np.isnan(lo):
        return f"{mean:.1f}", len(series)
    return f"{mean:.1f} ({lo:.1f}, {hi:.1f})", len(series)

if len(time_df) > 0:
    rows = []

    # Overall
    val, n = fmt_mean_ci(time_df["time_seconds"])
    rows.append({"Subset": "Overall", "Mean (95% CI) (s)": val, "N": n})

    for gt in ["simulated", "real"]:
        sub = time_df[time_df["ground_truth"] == gt]
        val, n = fmt_mean_ci(sub["time_seconds"])
        rows.append({"Subset": f"Ground truth: {gt}", "Mean (95% CI) (s)": val, "N": n})

    for correct in [True, False]:
        sub = time_df[time_df["correct"] == correct]
        label = "Correct Prediction" if correct else "Incorrect Prediction"
        val, n = fmt_mean_ci(sub["time_seconds"])
        rows.append({"Subset": label, "Mean (95% CI) (s)": val, "N": n})

    for cls in ["simulated", "real"]:
        sub = time_df[time_df["classification"] == cls]
        label = "AI Patient" if cls == "simulated" else "Real Patient"
        val, n = fmt_mean_ci(sub["time_seconds"])
        rows.append({"Subset": f"Classified: {label}", "Mean (95% CI) (s)": val, "N": n})

    display(pd.DataFrame(rows))

    # Per-rater breakdown
    raters = sorted(time_df["rater_id"].unique(),
                    key=lambda r: int(r.rsplit("_", 1)[-1]) if r.rsplit("_", 1)[-1].isdigit() else 0)
    rater_rows = []
    for rater in raters:
        rd = time_df[time_df["rater_id"] == rater]
        val, n = fmt_mean_ci(rd["time_seconds"])
        n_correct = int(rd["correct"].sum())
        acc = n_correct / n if n else 0
        rater_rows.append({"Rater": rater, "Mean (95% CI) (s)": val, "N": n, "Accuracy": f"{n_correct}/{n} = {acc:.0%}"})

    print("\nPer-rater:")
    display(pd.DataFrame(rater_rows))
else:
    print("No time data available.")

## Which Parameters Fool Humans?

For each sim parameter value, we compute the rate at which raters were **fooled** (i.e. classified the conversation incorrectly). Analysed separately for simulated conversations (fooled = classified as real) and real conversations (fooled = classified as simulated).

In [ ]:
# Load extracted params and merge with ratings
extracted = {}
with open(Path("..") / "data" / "extracted_params.jsonl") as f:
    for line in f:
        d = json.loads(line)
        extracted[d["session_id"]] = d

PARAM_COLS = [
    "anxiety", "frustration", "patience", "trust", "embarrassment",
    "recall_level", "oversharing_tendency",
    "health_literacy", "verbosity", "communication_style",
    "english_proficiency", "profanity_level", "typo_rate",
    "abbreviation_tendency", "punctuation_style", "capitalisation",
]

param_rows = []
for _, row in df.iterrows():
    cid = row["conversation_id"]
    if cid in extracted:
        sp = extracted[cid]["sim_params"]
        entry = {col: sp.get(col) for col in PARAM_COLS}
        entry["age"] = extracted[cid].get("age")
        entry["gender"] = extracted[cid].get("gender")
        entry["rater_id"] = row["rater_id"]
        entry["conversation_id"] = cid
        entry["ground_truth"] = row["ground_truth"]
        entry["correct"] = row["correct"]
        entry["fooled"] = not row["correct"]
        param_rows.append(entry)

pdf = pd.DataFrame(param_rows)
print(f"Merged {len(pdf)} ratings with extracted params")
pdf.head(3)

In [ ]:
def fooled_rate_by_param(data, param):
    rows = []
    for val, grp in data.groupby(param):
        n = len(grp)
        k = int(grp["fooled"].sum())
        rate = k / n if n else 0
        rows.append({"value": val, "n": n, "fooled": k, "rate": rate})
    return pd.DataFrame(rows).sort_values("rate", ascending=False)

analysis_params = [p for p in PARAM_COLS if pdf[p].nunique() > 1]

for gt_label, gt_value, description in [
    ("Simulated", "simulated", "fooled = rater said 'real' (simulation passed)"),
    ("Real", "real", "fooled = rater said 'simulated' (false accusation)"),
]:
    subset = pdf[pdf["ground_truth"] == gt_value]
    print(f"\n{'='*70}")
    print(f"{gt_label} conversations (N={len(subset)})  —  {description}")
    print(f"{'='*70}")

    for param in analysis_params:
        tbl = fooled_rate_by_param(subset, param)
        if len(tbl) < 2:
            continue
        print(f"\n  {param}:")
        for _, r in tbl.iterrows():
            bar = "█" * int(r["rate"] * 20)
            print(f"    {str(r['value']):<20s}  {r['fooled']:>2.0f}/{r['n']:<2.0f} = {r['rate']:>4.0%}  {bar}")

In [ ]:
# Visual: fooled rate by parameter value, split by ground truth
fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(analysis_params) * 0.6)), sharey=False)

for ax, (gt_label, gt_value) in zip(axes, [("Simulated", "simulated"), ("Real", "real")]):
    subset = pdf[pdf["ground_truth"] == gt_value]
    plot_data = []
    for param in analysis_params:
        for val, grp in subset.groupby(param):
            n = len(grp)
            k = int(grp["fooled"].sum())
            rate = k / n if n else 0
            plot_data.append({"param": param, "value": val, "rate": rate, "n": n})
    pdd = pd.DataFrame(plot_data)

    y_labels = []
    y_pos = []
    pos = 0
    for param in analysis_params:
        param_data = pdd[pdd["param"] == param].sort_values("rate", ascending=True)
        for _, r in param_data.iterrows():
            ax.barh(pos, r["rate"], height=0.6, color=SIM_COLOR if gt_value == "simulated" else REAL_COLOR, alpha=0.7)
            ax.text(r["rate"] + 0.02, pos, f"{r['rate']:.0%} (n={r['n']})", va="center", fontsize=7)
            y_labels.append(f"{param}: {r['value']}")
            y_pos.append(pos)
            pos += 1
        pos += 0.5

    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=8)
    ax.set_xlim(0, 1.1)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.axvline(0.5, color="grey", linestyle=":", linewidth=0.8, alpha=0.4)
    ax.set_xlabel("Fooled rate")
    ax.set_title(f"{gt_label} conversations", fontsize=12)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

fig.suptitle("Fooled Rate by Parameter Value", fontsize=14, y=1.01)
plt.tight_layout()

plt.savefig(EXPORT_DIR / "human_fooled_by_param.pdf", bbox_inches="tight", dpi=150)
plt.show()

### Parameter Deep Dives

For each parameter (or group), we show **accuracy** (% correctly classified) by parameter value — separately for simulated conversations (accuracy = detected as simulated) and real conversations (accuracy = recognised as real). Higher is always better.

In [ ]:
# Derived features and reusable plot helper
pdf["classified_simulated"] = pdf["correct"] != (pdf["ground_truth"] == "real")

pdf["n_nonstandard"] = (
    (pdf["typo_rate"] == "yes").astype(int) +
    (pdf["abbreviation_tendency"] == "yes").astype(int) +
    (pdf["punctuation_style"] == "none").astype(int) +
    (pdf["capitalisation"] == "all_lower").astype(int)
)

pdf["n_heightened"] = (
    (pdf["anxiety"] == "anxious").astype(int) +
    (pdf["frustration"] == "yes").astype(int) +
    (pdf["patience"] == "impatient").astype(int) +
    (pdf["embarrassment"] == "embarrassed").astype(int)
)

def plot_param_deep_dive(data, param, order=None, title="", xlabel=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

    for ax, (subtitle, gt_value, color) in zip(axes, [
        ("Simulated conversations\n(accuracy = detected as simulated)", "simulated", SIM_COLOR),
        ("Real conversations\n(accuracy = recognised as real)", "real", REAL_COLOR),
    ]):
        subset = data[data["ground_truth"] == gt_value]
        vals = order if order else sorted(subset[param].unique())
        vals = [v for v in vals if v in subset[param].values]
        x = np.arange(len(vals))

        rates, ns = [], []
        for v in vals:
            grp = subset[subset[param] == v]
            n = len(grp)
            k = int(grp["correct"].sum())
            rate = k / n if n else 0
            rates.append(rate)
            ns.append(n)

        ax.bar(x, rates, color=color, alpha=0.7, width=0.6)

        for i, (r, n) in enumerate(zip(rates, ns)):
            ax.text(i, r + 0.02, f"{r:.0%}\n(n={n})", ha="center", va="bottom", fontsize=8)

        ax.set_xticks(x)
        ax.set_xticklabels([str(v).replace("_", " ") for v in vals], fontsize=9)
        ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.8, alpha=0.4)
        ax.set_ylim(0, 1.05)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
        ax.set_title(subtitle, fontsize=10)
        for spine in ["top", "right"]:
            ax.spines[spine].set_visible(False)

    axes[0].set_ylabel("Accuracy")
    if xlabel:
        for ax in axes:
            ax.set_xlabel(xlabel)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# 1. Verbosity
VERBOSITY_ORDER = ["minimal", "very_brief", "brief", "moderate", "verbose"]
v_order = [v for v in VERBOSITY_ORDER if v in pdf["verbosity"].values]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

for ax, (subtitle, gt_value, color) in zip(axes, [
    ("Simulated conversations\n(accuracy = detected as simulated)", "simulated", SIM_COLOR),
    ("Real conversations\n(accuracy = recognised as real)", "real", REAL_COLOR),
]):
    subset = pdf[pdf["ground_truth"] == gt_value]
    x = np.arange(len(v_order))
    rates, ns = [], []
    for v in v_order:
        grp = subset[subset["verbosity"] == v]
        n = len(grp)
        k = int(grp["correct"].sum())
        rates.append(k / n if n else 0)
        ns.append(n)

    ax.plot(x, rates, "-o", color=color, markersize=7, linewidth=2, zorder=3)

    for i, (r, n) in enumerate(zip(rates, ns)):
        ax.text(i, r + 0.03, f"{r:.0%}\n(n={n})", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels([v.replace("_", " ") for v in v_order], fontsize=9)
    ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.8, alpha=0.4)
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.set_title(subtitle, fontsize=10)
    ax.set_xlabel("Verbosity level")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

axes[0].set_ylabel("Accuracy")
fig.suptitle("Verbosity", fontsize=13)
plt.tight_layout()
plt.show()

print("Higher verbosity makes simulated conversations easier to detect,")
print("but also makes real conversations harder to recognise as real.")

In [ ]:
# 2. Non-standard communication (aggregated: typos, abbreviations, no punctuation, all-lowercase)
pdf["has_nonstandard"] = pdf["n_nonstandard"].apply(lambda x: "at least one" if x >= 1 else "none")
plot_param_deep_dive(
    pdf, "has_nonstandard", order=["none", "at least one"],
    title="Non-Standard Communication Features (typos + abbreviations + no punctuation + all-lowercase)",
    xlabel="Non-standard features present",
)
print("Components: typo_rate=yes, abbreviation_tendency=yes, punctuation_style=none, capitalisation=all_lower")

In [ ]:
# 3. Recall level
plot_param_deep_dive(
    pdf, "recall_level", order=["low", "normal", "high"],
    title="Recall Level",
    xlabel="Recall level",
)

In [ ]:
# 4. Emotional state (aggregated: anxiety, frustration, impatience, embarrassment)
pdf["has_heightened"] = pdf["n_heightened"].apply(lambda x: "at least one" if x >= 1 else "none")
plot_param_deep_dive(
    pdf, "has_heightened", order=["none", "at least one"],
    title="Heightened Emotional State (anxiety + frustration + impatience + embarrassment)",
    xlabel="Heightened emotions present",
)
print("Components: anxiety=anxious, frustration=yes, patience=impatient, embarrassment=embarrassed")

In [ ]:
# 5. Health literacy
plot_param_deep_dive(
    pdf, "health_literacy", order=["low", "medium", "high"],
    title="Health Literacy",
    xlabel="Health literacy level",
)

## Which Parameters Make Humans Predict "Simulated"?

For each sim parameter value, we report the **% of ratings predicted as simulated** and **% predicted as real**. Analysed separately for simulated conversations and real conversations to reveal parameter-driven prediction biases.

In [ ]:
def pred_sim_rate_by_param(data, param):
    rows = []
    for val, grp in data.groupby(param):
        n = len(grp)
        k = int(grp["classified_simulated"].sum())
        rate = k / n if n else 0
        rows.append({"value": val, "n": n, "pred_sim": k, "rate": rate})
    return pd.DataFrame(rows).sort_values("rate", ascending=False)

for gt_label, gt_value, description in [
    ("Simulated", "simulated", "% predicted simulated (correct = high) vs predicted real (fooled = high)"),
    ("Real", "real", "% predicted simulated (false accusation = high) vs predicted real (correct = high)"),
]:
    subset = pdf[pdf["ground_truth"] == gt_value]
    print(f"\n{'='*70}")
    print(f"{gt_label} conversations (N={len(subset)})  —  {description}")
    print(f"{'='*70}")

    for param in analysis_params:
        tbl = pred_sim_rate_by_param(subset, param)
        if len(tbl) < 2:
            continue
        print(f"\n  {param}:")
        for _, r in tbl.iterrows():
            bar_sim = "█" * int(r["rate"] * 20)
            bar_real = "░" * int((1 - r["rate"]) * 20)
            print(f"    {str(r['value']):<20s}  sim {r['pred_sim']:>2.0f}/{r['n']:<2.0f} = {r['rate']:>4.0%} {bar_sim}  |  real {r['n'] - r['pred_sim']:>2.0f}/{r['n']:<2.0f} = {1 - r['rate']:>4.0%} {bar_real}")

In [ ]:
# Visual: % predicted simulated by parameter value, split by ground truth
fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(analysis_params) * 0.6)), sharey=False)

for ax, (gt_label, gt_value) in zip(axes, [("Simulated", "simulated"), ("Real", "real")]):
    subset = pdf[pdf["ground_truth"] == gt_value]
    plot_data = []
    for param in analysis_params:
        for val, grp in subset.groupby(param):
            n = len(grp)
            k = int(grp["classified_simulated"].sum())
            rate = k / n if n else 0
            plot_data.append({"param": param, "value": val, "rate": rate, "n": n})
    pdd = pd.DataFrame(plot_data)

    y_labels = []
    y_pos = []
    pos = 0
    for param in analysis_params:
        param_data = pdd[pdd["param"] == param].sort_values("rate", ascending=True)
        for _, r in param_data.iterrows():
            ax.barh(pos, r["rate"], height=0.6,
                    color=SIM_COLOR if gt_value == "simulated" else REAL_COLOR, alpha=0.7)
            ax.text(r["rate"] + 0.02, pos, f"{r['rate']:.0%} (n={r['n']})", va="center", fontsize=7)
            y_labels.append(f"{param}: {r['value']}")
            y_pos.append(pos)
            pos += 1
        pos += 0.5

    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=8)
    ax.set_xlim(0, 1.1)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.axvline(0.5, color="grey", linestyle=":", linewidth=0.8, alpha=0.4)
    ax.set_xlabel("% predicted simulated")
    ax.set_title(f"{gt_label} conversations", fontsize=12)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

fig.suptitle("Prediction Distribution by Parameter Value\n"
             "(% of ratings where humans predicted 'simulated')", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Overall: % predicted simulated by parameter value (all conversations combined)
fig, ax = plt.subplots(figsize=(10, max(6, len(analysis_params) * 0.6)))

plot_data = []
for param in analysis_params:
    for val, grp in pdf.groupby(param):
        n = len(grp)
        k = int(grp["classified_simulated"].sum())
        rate = k / n if n else 0
        plot_data.append({"param": param, "value": val, "rate": rate, "n": n})
pdd = pd.DataFrame(plot_data)

y_labels = []
y_pos = []
pos = 0
for param in analysis_params:
    param_data = pdd[pdd["param"] == param].sort_values("rate", ascending=True)
    for _, r in param_data.iterrows():
        ax.barh(pos, r["rate"], height=0.6, color="#8E44AD", alpha=0.7)
        ax.text(r["rate"] + 0.02, pos, f"{r['rate']:.0%} (n={r['n']})", va="center", fontsize=7)
        y_labels.append(f"{param}: {r['value']}")
        y_pos.append(pos)
        pos += 1
    pos += 0.5

ax.set_yticks(y_pos)
ax.set_yticklabels(y_labels, fontsize=8)
ax.set_xlim(0, 1.1)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.axvline(0.5, color="grey", linestyle=":", linewidth=0.8, alpha=0.4)
ax.set_xlabel("% predicted simulated")
ax.set_title("Overall Prediction Distribution by Parameter Value\n"
             "(all conversations — simulated + real combined)", fontsize=13)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

### Prediction Distribution Deep Dives

For each parameter (or group), we show **% predicted as simulated** by parameter value — separately for simulated and real conversations. For simulated conversations, higher means more detected; for real conversations, higher means more false accusations.

In [ ]:
def plot_pred_sim_deep_dive(data, param, order=None, title="", xlabel=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

    for ax, (subtitle, gt_value, color) in zip(axes, [
        ("Simulated conversations\n(% predicted simulated = detection rate)", "simulated", SIM_COLOR),
        ("Real conversations\n(% predicted simulated = false accusation rate)", "real", REAL_COLOR),
    ]):
        subset = data[data["ground_truth"] == gt_value]
        vals = order if order else sorted(subset[param].unique())
        vals = [v for v in vals if v in subset[param].values]
        x = np.arange(len(vals))

        rates, ns = [], []
        for v in vals:
            grp = subset[subset[param] == v]
            n = len(grp)
            k = int(grp["classified_simulated"].sum())
            rate = k / n if n else 0
            rates.append(rate)
            ns.append(n)

        ax.bar(x, rates, color=color, alpha=0.7, width=0.6)

        for i, (r, n) in enumerate(zip(rates, ns)):
            ax.text(i, r + 0.02, f"{r:.0%}\n(n={n})", ha="center", va="bottom", fontsize=8)

        ax.set_xticks(x)
        ax.set_xticklabels([str(v).replace("_", " ") for v in vals], fontsize=9)
        ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.8, alpha=0.4)
        ax.set_ylim(0, 1.05)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
        ax.set_title(subtitle, fontsize=10)
        for spine in ["top", "right"]:
            ax.spines[spine].set_visible(False)

    axes[0].set_ylabel("% predicted simulated")
    if xlabel:
        for ax in axes:
            ax.set_xlabel(xlabel)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# 1. Verbosity
v_order = [v for v in VERBOSITY_ORDER if v in pdf["verbosity"].values]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

for ax, (subtitle, gt_value, color) in zip(axes, [
    ("Simulated conversations\n(% predicted simulated = detection rate)", "simulated", SIM_COLOR),
    ("Real conversations\n(% predicted simulated = false accusation rate)", "real", REAL_COLOR),
]):
    subset = pdf[pdf["ground_truth"] == gt_value]
    x = np.arange(len(v_order))
    rates, ns = [], []
    for v in v_order:
        grp = subset[subset["verbosity"] == v]
        n = len(grp)
        k = int(grp["classified_simulated"].sum())
        rates.append(k / n if n else 0)
        ns.append(n)

    ax.plot(x, rates, "-o", color=color, markersize=7, linewidth=2, zorder=3)

    for i, (r, n) in enumerate(zip(rates, ns)):
        ax.text(i, r + 0.03, f"{r:.0%}\n(n={n})", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels([v.replace("_", " ") for v in v_order], fontsize=9)
    ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.8, alpha=0.4)
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.set_title(subtitle, fontsize=10)
    ax.set_xlabel("Verbosity level")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

axes[0].set_ylabel("% predicted simulated")
fig.suptitle("Verbosity: Prediction Distribution", fontsize=13)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "human_verbosity_by_gt.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Verbosity: all conversations combined
v_order = [v for v in VERBOSITY_ORDER if v in pdf["verbosity"].values]

fig, ax = plt.subplots(figsize=(8, 4.5))

x = np.arange(len(v_order))
rates, ns = [], []
for v in v_order:
    grp = pdf[pdf["verbosity"] == v]
    n = len(grp)
    k = int(grp["classified_simulated"].sum())
    rates.append(k / n if n else 0)
    ns.append(n)

ax.plot(x, rates, "-o", color="#8E44AD", markersize=7, linewidth=2, zorder=3)

for i, (r, n) in enumerate(zip(rates, ns)):
    ax.text(i, r + 0.03, f"{r:.0%}\n(n={n})", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([v.replace("_", " ") for v in v_order], fontsize=9)
ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.8, alpha=0.4)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.set_ylabel("% predicted simulated")
ax.set_xlabel("Verbosity level")
ax.set_title("Verbosity: Prediction Distribution (All Conversations)", fontsize=13)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(EXPORT_DIR / "human_verbosity_combined.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# 2. Non-standard communication
plot_pred_sim_deep_dive(
    pdf, "has_nonstandard", order=["none", "at least one"],
    title="Non-Standard Communication Features: Prediction Distribution",
    xlabel="Non-standard features present",
)

In [ ]:
# 3. Recall level
plot_pred_sim_deep_dive(
    pdf, "recall_level", order=["low", "normal", "high"],
    title="Recall Level: Prediction Distribution",
    xlabel="Recall level",
)

In [ ]:
# 4. Emotional state
plot_pred_sim_deep_dive(
    pdf, "has_heightened", order=["none", "at least one"],
    title="Heightened Emotional State: Prediction Distribution",
    xlabel="Heightened emotions present",
)

In [ ]:
# 5. Health literacy
plot_pred_sim_deep_dive(
    pdf, "health_literacy", order=["low", "medium", "high"],
    title="Health Literacy: Prediction Distribution",
    xlabel="Health literacy level",
)

## Qualitative Feedback

In [ ]:
feedback_df = df[df["feedback"].str.strip().astype(bool)].copy()
if len(feedback_df) > 0:
    feedback_df["correct_label"] = feedback_df["correct"].map({True: "correct", False: "wrong"})
    cols = ["rater_id", "ground_truth", "classification", "correct_label", "feedback"]
    with pd.option_context("display.max_colwidth", None):
        display(
            feedback_df[cols].style
            .set_properties(subset=["feedback"], **{"text-align": "left"})
            .set_table_styles([{"selector": "th.col4", "props": [("text-align", "left")]}])
        )
else:
    print("No qualitative feedback provided yet.")

In [ ]:
# Feedback frequency: when do raters leave feedback?
has_feedback = df["feedback"].str.strip().astype(bool)

fb_rows = []
for label, mask in [
    ("All ratings", pd.Series(True, index=df.index)),
    ("Correct predictions", df["correct"] == True),
    ("Wrong predictions", df["correct"] == False),
    ("Actually simulated", df["ground_truth"] == "simulated"),
    ("Actually real", df["ground_truth"] == "real"),
]:
    n = int(mask.sum())
    n_fb = int((has_feedback & mask).sum())
    fb_rows.append({
        "Subset": label,
        "N": n,
        "With feedback": n_fb,
        "Without feedback": n - n_fb,
        "% with feedback": f"{n_fb}/{n} = {n_fb/n:.0%}" if n else "-",
    })

display(pd.DataFrame(fb_rows))

# Cross-tab: correct x ground_truth
print("\nFeedback rate by correctness × ground truth:")
cross_rows = []
for correct_label, correct_val in [("Correct", True), ("Wrong", False)]:
    for gt in ["simulated", "real"]:
        mask = (df["correct"] == correct_val) & (df["ground_truth"] == gt)
        n = int(mask.sum())
        n_fb = int((has_feedback & mask).sum())
        cross_rows.append({
            "Prediction": correct_label,
            "Ground truth": gt,
            "N": n,
            "With feedback": n_fb,
            "% with feedback": f"{n_fb/n:.0%}" if n else "-",
        })

display(pd.DataFrame(cross_rows))

interesting to see that still a lot of wrong predictions get feedback

## Human vs LLM Inter-Rater Agreement

Each conversation was rated once by a human and once by each LLM judge (claude-opus-4.6 and gpt-5.4) under the same sequential rater protocol. We match ratings by rater index and conversation ID to compute pairwise agreement.

In [ ]:
# Load LLM sequential ratings
SEQUENTIAL_FILES = {
    "claude-opus-4.6": Path("..") / "llms" / "results" / "runs" / "20260518_170302_sequential" / "sequential_claude-opus-4.6.json",
    "gpt-5.4": Path("..") / "llms" / "results" / "runs" / "20260601_165135_sequential" / "sequential_gpt-5.4.json",
}

llm_ratings = []
for judge_name, path in SEQUENTIAL_FILES.items():
    with open(path) as f:
        sessions = json.load(f)
    for session in sessions:
        for r in session["ratings"]:
            rater_idx = int(r["rater_id"].rsplit("_", 1)[-1])
            llm_ratings.append({
                "judge": judge_name,
                "rater_index": rater_idx,
                "conversation_id": r["conversation_id"],
                "ground_truth": r["ground_truth"],
                "classification": r["classification"],
                "classified_simulated": r["classified_simulated"],
                "correct": r["correct"],
            })

llm_df = pd.DataFrame(llm_ratings)

# Filter to rater indices that have human ratings
human_indices = set(df["rater_index"].unique())
llm_df = llm_df[llm_df["rater_index"].isin(human_indices)]

print(f"LLM ratings matching human rater indices: {len(llm_df)}")
print(f"  Rater indices: {sorted(human_indices)}")
print(f"  Judges: {sorted(llm_df['judge'].unique())}")

In [ ]:
# Merge human and LLM ratings by conversation_id
human_slim = df[["conversation_id", "rater_index", "ground_truth", "classification", "classified_simulated", "correct"]].copy()
human_slim = human_slim.rename(columns={
    "classification": "human_classification",
    "classified_simulated": "human_sim",
    "correct": "human_correct",
})

agreement_rows = []
for judge in sorted(llm_df["judge"].unique()):
    jdf = llm_df[llm_df["judge"] == judge][["conversation_id", "rater_index", "classification", "classified_simulated", "correct"]].copy()
    jdf = jdf.rename(columns={
        "classification": "llm_classification",
        "classified_simulated": "llm_sim",
        "correct": "llm_correct",
    })
    merged = human_slim.merge(jdf, on=["conversation_id", "rater_index"], how="inner")
    merged["agree"] = merged["human_classification"] == merged["llm_classification"]
    merged["judge"] = judge
    agreement_rows.append(merged)

agree_df = pd.concat(agreement_rows, ignore_index=True)
print(f"Matched {len(agree_df)} conversation-level pairs across {agree_df['judge'].nunique()} LLM judges")

In [ ]:
# Cohen's kappa and agreement metrics
def cohens_kappa(y1, y2):
    n = len(y1)
    if n == 0:
        return np.nan
    po = np.mean(np.array(y1) == np.array(y2))
    p1 = np.mean(y1)
    p2 = np.mean(y2)
    pe = p1 * p2 + (1 - p1) * (1 - p2)
    if pe == 1.0:
        return 1.0
    return (po - pe) / (1 - pe)

summary_rows = []
for judge in sorted(agree_df["judge"].unique()):
    jdata = agree_df[agree_df["judge"] == judge]

    for gt_label, gt_filter in [("All", None), ("Simulated", "simulated"), ("Real", "real")]:
        subset = jdata if gt_filter is None else jdata[jdata["ground_truth"] == gt_filter]
        n = len(subset)
        n_agree = int(subset["agree"].sum())
        pct_agree = n_agree / n if n else 0
        kappa = cohens_kappa(subset["human_sim"].values, subset["llm_sim"].values)

        both_correct = int((subset["human_correct"] & subset["llm_correct"]).sum())
        both_wrong = int((~subset["human_correct"] & ~subset["llm_correct"]).sum())
        human_only = int((subset["human_correct"] & ~subset["llm_correct"]).sum())
        llm_only = int((~subset["human_correct"] & subset["llm_correct"]).sum())

        summary_rows.append({
            "LLM Judge": judge,
            "Subset": gt_label,
            "N": n,
            "Agreement": f"{n_agree}/{n} = {pct_agree:.0%}",
            "Cohen's κ": f"{kappa:.2f}",
            "Both correct": both_correct,
            "Both wrong": both_wrong,
            "Human only": human_only,
            "LLM only": llm_only,
        })

display(pd.DataFrame(summary_rows))

In [ ]:
# Agreement matrix: human classification vs LLM classification
from sklearn.metrics import cohen_kappa_score

fig, axes = plt.subplots(1, len(agree_df["judge"].unique()), figsize=(12, 4), sharey=True)
judges = sorted(agree_df["judge"].unique())
if len(judges) == 1:
    axes = [axes]

cmap = plt.cm.GnBu

for ax, judge in zip(axes, judges):
    jdata = agree_df[agree_df["judge"] == judge]

    labels = ["real", "simulated"]
    cm = np.zeros((2, 2), dtype=int)
    for i, h_cls in enumerate(labels):
        for j, l_cls in enumerate(labels):
            cm[i, j] = int(((jdata["human_classification"] == h_cls) & (jdata["llm_classification"] == l_cls)).sum())

    kappa = cohen_kappa_score(jdata["human_classification"], jdata["llm_classification"])
    agree_pct = (cm[0, 0] + cm[1, 1]) / cm.sum()

    im = ax.imshow(cm, cmap=cmap, vmin=0, vmax=cm.max() or 1, aspect="equal")
    for i in range(2):
        for j in range(2):
            row_total = cm[i].sum()
            pct = cm[i, j] / row_total * 100 if row_total else 0
            ax.text(j, i, f"{cm[i, j]}\n({pct:.0f}%)",
                    ha="center", va="center", fontsize=11,
                    color="white" if cm[i, j] > cm.max() * 0.6 else "black")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Real", "Simulated"], fontsize=10)
    ax.set_yticklabels(["Real", "Simulated"], fontsize=10)
    ax.set_xlabel(f"{judge} classification", fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel("Human classification", fontsize=10)
    ax.set_title(f"\u03ba = {kappa:.3f}, agreement = {agree_pct:.0%}", fontsize=10)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(left=False, bottom=False)

fig.suptitle("Human vs LLM Classification Agreement", fontsize=13)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "human_llm_agreement.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Side-by-side accuracy comparison (on the same conversations)
comp_rows = []

# Human accuracy on matched conversations
matched_convos = agree_df["conversation_id"].unique()
human_matched = df[df["conversation_id"].isin(matched_convos)]
n_h = len(human_matched)
k_h = int(human_matched["correct"].sum())
comp_rows.append({
    "Rater": "Human",
    "N": n_h,
    "Accuracy": f"{k_h}/{n_h} = {k_h/n_h:.0%}" if n_h else "-",
    "Det. Rate (sim)": None,
    "Real Recog. (real)": None,
})

sim_h = human_matched[human_matched["ground_truth"] == "simulated"]
real_h = human_matched[human_matched["ground_truth"] == "real"]
tp_h = int(sim_h["classified_simulated"].sum())
tn_h = int((~real_h["classified_simulated"]).sum())
comp_rows[0]["Det. Rate (sim)"] = f"{tp_h}/{len(sim_h)} = {tp_h/len(sim_h):.0%}" if len(sim_h) else "-"
comp_rows[0]["Real Recog. (real)"] = f"{tn_h}/{len(real_h)} = {tn_h/len(real_h):.0%}" if len(real_h) else "-"

# LLM accuracy on same conversations
for judge in sorted(llm_df["judge"].unique()):
    jdata = llm_df[(llm_df["judge"] == judge) & (llm_df["conversation_id"].isin(matched_convos))]
    n_j = len(jdata)
    k_j = int(jdata["correct"].sum())
    sim_j = jdata[jdata["ground_truth"] == "simulated"]
    real_j = jdata[jdata["ground_truth"] == "real"]
    tp_j = int(sim_j["classified_simulated"].sum())
    tn_j = int((~real_j["classified_simulated"]).sum())
    comp_rows.append({
        "Rater": judge,
        "N": n_j,
        "Accuracy": f"{k_j}/{n_j} = {k_j/n_j:.0%}" if n_j else "-",
        "Det. Rate (sim)": f"{tp_j}/{len(sim_j)} = {tp_j/len(sim_j):.0%}" if len(sim_j) else "-",
        "Real Recog. (real)": f"{tn_j}/{len(real_j)} = {tn_j/len(real_j):.0%}" if len(real_j) else "-",
    })

print("Accuracy on the same set of conversations:")
display(pd.DataFrame(comp_rows))

## Classification Bias: Proportion of "Simulated" Predictions

Do raters have a tendency to label conversations as simulated or real? A balanced rater should predict ~50% simulated given the 50/50 base rate.

In [ ]:
# Proportion of predictions that are "simulated" — humans vs LLMs
bias_rows = []

# Human
n_human = len(df)
n_human_sim = int(df["classified_simulated"].sum())
bias_rows.append({
    "Rater": "Human",
    "N": n_human,
    "Classified simulated": n_human_sim,
    "Classified real": n_human - n_human_sim,
    "% classified simulated": n_human_sim / n_human,
})

# LLMs (on the matched conversations)
for judge in sorted(llm_df["judge"].unique()):
    jdata = llm_df[llm_df["judge"] == judge]
    n_j = len(jdata)
    n_j_sim = int(jdata["classified_simulated"].sum())
    bias_rows.append({
        "Rater": judge,
        "N": n_j,
        "Classified simulated": n_j_sim,
        "Classified real": n_j - n_j_sim,
        "% classified simulated": n_j_sim / n_j,
    })

bias_df = pd.DataFrame(bias_rows)

# Table
display_bias = bias_df.copy()
display_bias["% classified simulated"] = display_bias["% classified simulated"].map(lambda x: f"{x:.0%}")
display(display_bias)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(bias_df))
width = 0.35
pct_sim = bias_df["% classified simulated"].values if isinstance(bias_df["% classified simulated"].iloc[0], float) else [r["Classified simulated"] / r["N"] for _, r in bias_df.iterrows()]
pct_real = [1 - p for p in pct_sim]

ax.bar(x - width / 2, pct_sim, width, color=SIM_COLOR, label="Classified simulated")
ax.bar(x + width / 2, pct_real, width, color=REAL_COLOR, label="Classified real")

for i, (ps, pr) in enumerate(zip(pct_sim, pct_real)):
    ax.text(x[i] - width / 2, ps + 0.02, f"{ps:.0%}", ha="center", fontsize=9)
    ax.text(x[i] + width / 2, pr + 0.02, f"{pr:.0%}", ha="center", fontsize=9)

ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8, alpha=0.5, label="Balanced (50/50)")
ax.set_xticks(x)
ax.set_xticklabels(bias_df["Rater"], fontsize=10)
ax.set_ylabel("Proportion of predictions")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.set_ylim(0, 1.05)
ax.legend(loc="upper left", fontsize=9)
ax.set_title("Classification Tendencies by Rater Type", fontsize=12)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()

plt.savefig(EXPORT_DIR / "human_classification_bias.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
import statsmodels.formula.api as smf

# Linear probability model: fooled ~ simulation params, clustered by rater
pdf["fooled_int"] = pdf["fooled"].astype(int)

# Use params with >1 unique level
model_params = [p for p in PARAM_COLS if pdf[p].nunique() > 1]

# Drop params with any level that has <3 observations (unstable estimates)
stable_params = []
for p in model_params:
    min_count = pdf[p].value_counts().min()
    if min_count >= 3:
        stable_params.append(p)
    else:
        print(f"Dropping {p}: min group size = {min_count}")

formula_terms = " + ".join([f"C({p})" for p in stable_params])
formula = f"fooled_int ~ C(ground_truth) + {formula_terms}"
print(f"\nFormula: {formula}\n")

model = smf.ols(formula, data=pdf).fit(
    cov_type="cluster",
    cov_kwds={"groups": pdf["rater_id"]},
)
print(model.summary())

# Tidy coefficient table
coef_df = pd.DataFrame({
    "coef": model.params,
    "se": model.bse,
    "t": model.tvalues,
    "p": model.pvalues,
    "ci_lo": model.conf_int()[0],
    "ci_hi": model.conf_int()[1],
}).round(3)

print("\nCoefficients (sorted by absolute effect size):")
display(coef_df.reindex(coef_df["coef"].abs().sort_values(ascending=False).index))

sadly sample size is too small. should fix the neutral option etc. but not investing on this for now

## Export Metrics for Report

In [ ]:
EXPORT_DIR = Path("..") / "report" / "data"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

r = results
human_export = {k: (int(v) if isinstance(v, (np.integer, int)) and not isinstance(v, bool) else
                    float(v) if isinstance(v, (np.floating, float)) and not isinstance(v, bool) else
                    bool(v) if isinstance(v, (np.bool_, bool)) else v)
                for k, v in r.items()}

with open(EXPORT_DIR / "human_metrics.json", "w") as f:
    json.dump(human_export, f, indent=2)

print(f"Exported to {EXPORT_DIR.resolve()}:")
for p in sorted(EXPORT_DIR.glob("human_*.json")):
    print(f"  {p.name}")

# --- Time metrics ---
time_df_export = df.dropna(subset=["time_seconds"])
time_export = {}
if len(time_df_export) > 0:
    time_export["overall_mean"] = float(time_df_export["time_seconds"].mean())
    time_export["overall_n"] = len(time_df_export)
    for gt in ["simulated", "real"]:
        sub = time_df_export[time_df_export["ground_truth"] == gt]
        if len(sub):
            time_export[f"{gt}_mean"] = float(sub["time_seconds"].mean())
            time_export[f"{gt}_n"] = len(sub)
    for corr_label, corr_val in [("correct", True), ("wrong", False)]:
        sub = time_df_export[time_df_export["correct"] == corr_val]
        if len(sub):
            time_export[f"{corr_label}_mean"] = float(sub["time_seconds"].mean())
            time_export[f"{corr_label}_n"] = len(sub)

    with open(EXPORT_DIR / "human_time_metrics.json", "w") as f:
        json.dump(time_export, f, indent=2)
    print(f"  human_time_metrics.json")

# --- Feedback summary ---
has_feedback = df["feedback"].str.strip().astype(bool)
fb_export = {
    "total_ratings": len(df),
    "ratings_with_feedback": int(has_feedback.sum()),
    "feedback_rate": float(has_feedback.mean()),
    "correct_with_feedback": int((has_feedback & (df["correct"] == True)).sum()),
    "correct_total": int((df["correct"] == True).sum()),
    "wrong_with_feedback": int((has_feedback & (df["correct"] == False)).sum()),
    "wrong_total": int((df["correct"] == False).sum()),
}
with open(EXPORT_DIR / "human_feedback_summary.json", "w") as f:
    json.dump(fb_export, f, indent=2)
print(f"  human_feedback_summary.json")


# --- Feedback table ---
has_feedback = df["feedback"].str.strip().astype(bool)
feedback_df = df[has_feedback].copy()
if len(feedback_df) > 0:
    fb_table = []
    for _, row in feedback_df.iterrows():
        fb_table.append({
            "rater_id": row["rater_id"],
            "ground_truth": row["ground_truth"],
            "classification": row["classification"],
            "correct": bool(row["correct"]),
            "feedback": row["feedback"].strip(),
        })
    with open(EXPORT_DIR / "human_feedback_table.json", "w") as f:
        json.dump(fb_table, f, indent=2, ensure_ascii=False)
    print(f"  human_feedback_table.json ({len(fb_table)} entries)")


In [ ]:
# --- Combined sequential protocol table (human + LLM) for Section 5 ---
combined_rows = []

# Human
h = results
combined_rows.append({
    "judge": "Human",
    "n_raters": int(h["n_raters"]),
    "tp": int(h["tp"]), "fn": int(h["fn"]), "fp": int(h["fp"]), "tn": int(h["tn"]),
})

# LLM sequential judges
with open(EXPORT_DIR / "llm_sequential_metrics.json") as f:
    seq_metrics = json.load(f)

for s in seq_metrics:
    combined_rows.append({
        "judge": str(s["label"]),
        "n_raters": int(s["n_raters"]),
        "tp": int(s["tp"]), "fn": int(s["fn"]), "fp": int(s["fp"]), "tn": int(s["tn"]),
    })

with open(EXPORT_DIR / "combined_sequential_metrics.json", "w") as f:
    json.dump(combined_rows, f, indent=2)

print("Combined sequential metrics:")
for r in combined_rows:
    n_sim = r["tp"] + r["fn"]
    n_real = r["tn"] + r["fp"]
    n_total = n_sim + n_real
    det = r["tp"] / n_sim if n_sim else 0
    rr = r["tn"] / n_real if n_real else 0
    acc = (r["tp"] + r["tn"]) / n_total if n_total else 0
    print(f"  {r["judge"]}: det={det:.0%}, real_recog={rr:.0%}, acc={acc:.0%}")

print(f"\nExported: combined_sequential_metrics.json")
